In [1]:

import pandas as pd
import numpy as np
import re

invoices = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\invoices.csv")
payments = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\gateway_payments.csv")


Q2_START = pd.Timestamp("2025-04-01", tz="Asia/Kolkata")
Q2_END = pd.Timestamp("2025-06-30 23:59:59.999999", tz="Asia/Kolkata")
Q3_END = pd.Timestamp("2025-09-30 23:59:59.999999", tz="Asia/Kolkata")


def normalize_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["match_key"] = invoices["invoice_id"].apply(normalize_id)
payments["match_key"] = payments["invoice_ref"].apply(normalize_id)


invoices["invoice_date"] = pd.to_datetime(
    invoices["invoice_date"], errors="coerce"
)


if invoices["invoice_date"].dt.tz is None:
    invoices["invoice_date"] = (
        invoices["invoice_date"].dt.tz_localize("Asia/Kolkata")
    )
else:
    invoices["invoice_date"] = (
        invoices["invoice_date"].dt.tz_convert("Asia/Kolkata")
    )


if "created_at_ist" in payments.columns:
    payments["payment_date"] = pd.to_datetime(
        payments["created_at_ist"], errors="coerce"
    )
    if payments["payment_date"].dt.tz is None:
        payments["payment_date"] = (
            payments["payment_date"].dt.tz_localize("Asia/Kolkata")
        )
    else:
        payments["payment_date"] = (
            payments["payment_date"].dt.tz_convert("Asia/Kolkata")
        )

elif "created_at_epoch" in payments.columns:
    payments["payment_date"] = pd.to_datetime(
        pd.to_numeric(payments["created_at_epoch"], errors="coerce"),
        unit="s",
        utc=True
    ).dt.tz_convert("Asia/Kolkata")

else:
    payments["payment_date"] = pd.to_datetime(
        payments["created_at_utc"], errors="coerce", utc=True
    ).dt.tz_convert("Asia/Kolkata")


q2_invoices = invoices.loc[
    invoices["invoice_date"].between(Q2_START, Q2_END)
].copy()

q2_invoices["amount_rs"] = pd.to_numeric(
    q2_invoices["amount_rs"], errors="coerce"
).fillna(0)


payments["amount_rs"] = pd.to_numeric(
    payments["amount_rs"], errors="coerce"
)

captured = payments.loc[
    payments["status"].astype(str).str.strip().str.lower()
    .eq("captured")
].copy()


if "payment_id" in captured.columns:
    captured = captured.drop_duplicates(
        subset=["payment_id"], keep="first"
    )

q2_keys = set(q2_invoices["match_key"].dropna())
linked = captured.loc[
    captured["match_key"].isin(q2_keys)
    & captured["payment_date"].notna()
].copy()


paid_by_q2 = (
    linked.loc[linked["payment_date"] <= Q2_END]
    .groupby("match_key")["amount_rs"]
    .sum()
    .rename("paid_by_q2_rs")
)


paid_in_q3 = (
    linked.loc[
        (linked["payment_date"] > Q2_END)
        & (linked["payment_date"] <= Q3_END)
    ]
    .groupby("match_key")["amount_rs"]
    .sum()
    .rename("paid_in_q3_rs")
)


timing = q2_invoices.copy()

timing = timing.join(paid_by_q2, on="match_key")
timing = timing.join(paid_in_q3, on="match_key")

timing["paid_by_q2_rs"] = timing["paid_by_q2_rs"].fillna(0)
timing["paid_in_q3_rs"] = timing["paid_in_q3_rs"].fillna(0)


timing["outstanding_at_q2_end_rs"] = (
    timing["amount_rs"] - timing["paid_by_q2_rs"]
).clip(lower=0)


timing["later_payment_applied_rs"] = np.minimum(
    timing["outstanding_at_q2_end_rs"],
    timing["paid_in_q3_rs"].clip(lower=0)
)


timing["still_outstanding_after_q3_rs"] = (
    timing["outstanding_at_q2_end_rs"]
    - timing["later_payment_applied_rs"]
)

timing["timing_status"] = np.select(
    [
        timing["outstanding_at_q2_end_rs"].eq(0),
        timing["still_outstanding_after_q3_rs"].eq(0)
    ],
    [
        "Paid by Q2 end",
        "Paid after Q2 during Q3"
    ],
    default="Still outstanding after Q3 check"
)

summary = pd.Series({
    "Q2 invoices (count)": len(timing),
    "Q2 invoiced amount (Rs)": timing["amount_rs"].sum(),
    "Outstanding at Q2 end (Rs)": (
        timing["outstanding_at_q2_end_rs"].sum()
    ),
    "Outstanding invoices at Q2 end (count)": (
        timing["outstanding_at_q2_end_rs"].gt(0).sum()
    ),
    "Later payments applied (Rs)": (
        timing["later_payment_applied_rs"].sum()
    ),
    "Invoices fully recovered during Q3 (count)": (
        (
            timing["outstanding_at_q2_end_rs"].gt(0)
            & timing["still_outstanding_after_q3_rs"].eq(0)
        ).sum()
    ),
    "Still outstanding after Q3 (Rs)": (
        timing["still_outstanding_after_q3_rs"].sum()
    ),
    "Still outstanding after Q3 (count)": (
        timing["still_outstanding_after_q3_rs"].gt(0).sum()
    )
})

print("\nTIMING ANALYSIS SUMMARY")
print(summary.to_string())


unpaid_at_q2_end = timing.loc[
    timing["outstanding_at_q2_end_rs"] > 0,
    [
        "invoice_id",
        "invoice_date",
        "amount_rs",
        "paid_by_q2_rs",
        "outstanding_at_q2_end_rs",
        "paid_in_q3_rs",
        "later_payment_applied_rs",
        "still_outstanding_after_q3_rs",
        "timing_status"
    ]
].sort_values("still_outstanding_after_q3_rs", ascending=False)

print("\nINVOICES UNPAID AT Q2 END")
print(unpaid_at_q2_end.head(20).to_string(index=False))



TIMING ANALYSIS SUMMARY
Q2 invoices (count)                               77915.0
Q2 invoiced amount (Rs)                       118105523.0
Outstanding at Q2 end (Rs)                      9088446.0
Outstanding invoices at Q2 end (count)             6065.0
Later payments applied (Rs)                     4407518.0
Invoices fully recovered during Q3 (count)         2990.0
Still outstanding after Q3 (Rs)                 4680928.0
Still outstanding after Q3 (count)                 3075.0

INVOICES UNPAID AT Q2 END
     invoice_id              invoice_date  amount_rs  paid_by_q2_rs  outstanding_at_q2_end_rs  paid_in_q3_rs  later_payment_applied_rs  still_outstanding_after_q3_rs                    timing_status
INV-2025-084920 2025-04-07 00:00:00+05:30     7997.0            0.0                    7997.0            0.0                       0.0                         7997.0 Still outstanding after Q3 check
INV-2025-117463 2025-05-15 00:00:00+05:30     7348.0            0.0                   

In [2]:

# Normalize payment statuses
payments["status_clean"] = (
    payments["status"].astype(str).str.strip().str.lower()
)

# 1. Find invoices with multiple distinct payment IDs
invoice_payment_counts = (
    payments.groupby("match_key")
    .agg(
        payment_attempts=("payment_id", "size"),
        distinct_payment_ids=("payment_id", "nunique"),
        statuses=("status_clean",
                  lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

multiple_attempts = invoice_payment_counts[
    invoice_payment_counts["distinct_payment_ids"] > 1
].copy()

print("Invoices with multiple payment IDs:",
      len(multiple_attempts))

print(multiple_attempts.head(20).to_string(index=False))


# 2. Check failed attempts followed by captured payments
payment_status_by_invoice = (
    payments.groupby("match_key")
    .agg(
        has_failed=("status_clean",
                    lambda x: x.eq("failed").any()),
        has_captured=("status_clean",
                      lambda x: x.eq("captured").any()),
        payment_ids=("payment_id",
                     lambda x: x.nunique()),
        total_rows=("payment_id", "size")
    )
    .reset_index()
)

retried_successfully = payment_status_by_invoice[
    payment_status_by_invoice["has_failed"]
    & payment_status_by_invoice["has_captured"]
].copy()

print("\nInvoices with both failed and captured attempts:",
      len(retried_successfully))

print(retried_successfully.head(20).to_string(index=False))


# 3. Find repeated payment IDs
duplicate_payment_rows = payments[
    payments.duplicated("payment_id", keep=False)
].sort_values("payment_id")

print("\nRows sharing a payment ID:",
      len(duplicate_payment_rows))

print(duplicate_payment_rows.head(30).to_string(index=False))


# 4. Summarize statuses and amounts
print("\nPayment status summary:")
print(
    payments.groupby("status_clean")
    .agg(
        row_count=("payment_id", "size"),
        distinct_payment_ids=("payment_id", "nunique"),
        amount_rs=("amount_rs", "sum")
    )
    .sort_values("row_count", ascending=False)
    .to_string()
)


Invoices with multiple payment IDs: 19814
    match_key  payment_attempts  distinct_payment_ids         statuses
INV2025000011                 2                     2 captured, failed
INV2025000089                 2                     2 captured, failed
INV2025000129                 2                     2 captured, failed
INV2025000148                 2                     2 captured, failed
INV2025000209                 2                     2 captured, failed
INV2025000248                 2                     2 captured, failed
INV2025000252                 2                     2 captured, failed
INV2025000269                 2                     2 captured, failed
INV2025000287                 2                     2 captured, failed
INV2025000363                 2                     2 captured, failed
INV2025000373                 2                     2 captured, failed
INV2025000398                 2                     2 captured, failed
INV2025000411                 2    

In [3]:

# Standardize payment status
payments["status_clean"] = (
    payments["status"].astype(str).str.strip().str.lower()
)

# Keep only successful/captured payment records
captured = payments[payments["status_clean"] == "captured"].copy()

# Count captured payment IDs and sum their amounts for each invoice
captured_by_invoice = (
    captured.groupby("match_key")
    .agg(
        captured_rows=("payment_id", "size"),
        captured_payment_ids=("payment_id", "nunique"),
        captured_amount_rs=("amount_rs", "sum"),
        payment_ids=("payment_id", lambda x: ", ".join(x.astype(str)))
    )
    .reset_index()
)

# Invoices with more than one distinct captured payment ID
multiple_captured = captured_by_invoice[
    captured_by_invoice["captured_payment_ids"] > 1
].sort_values("captured_amount_rs", ascending=False)

print("Invoices with multiple captured payment IDs:",
      len(multiple_captured))

print(multiple_captured.head(30).to_string(index=False))

print("\nTotal captured amount across these invoices (Rs):",
      multiple_captured["captured_amount_rs"].sum())


Invoices with multiple captured payment IDs: 1925
    match_key  captured_rows  captured_payment_ids  captured_amount_rs                    payment_ids
INV2026039887              2                     2             12117.0 pay_0000056130, pay_0000056139
INV2025086936              2                     2             10067.0 pay_000001582f, pay_000001584b
INV2026075063              2                     2              9617.0 pay_000005ec4d, pay_000005ec68
INV2025179470              2                     2              9597.0 pay_000002c6b3, pay_000002c6f1
INV2025177652              2                     2              9567.0 pay_000002bf9b, pay_000002bfb7
INV2025216693              2                     2              9467.0 pay_0000035a43, pay_0000035a7a
INV2025300385              2                     2              9267.0 pay_000004a55a, pay_000004a574
INV2026138476              2                     2              9168.0 pay_000006e756, pay_000006e760
INV2025150884              2    

In [5]:

# Total successful payments for each invoice
captured_totals = (
    payments[payments["status_clean"] == "captured"]
    .groupby("match_key")
    .agg(
        captured_payment_count=("payment_id", "nunique"),
        total_captured_rs=("amount_rs", "sum"),
        payment_ids=("payment_id", lambda x: ", ".join(x.astype(str)))
    )
    .reset_index()
)

# Keep one row per invoice and use its original billed amount
invoice_amounts = (
    invoices.groupby("match_key")
    .agg(invoice_amount_rs=("amount_rs", "first"))
    .reset_index()
)

# Compare captured payments against invoice value
payment_comparison = invoice_amounts.merge(
    captured_totals,
    on="match_key",
    how="left"
)

payment_comparison["total_captured_rs"] = (
    payment_comparison["total_captured_rs"].fillna(0)
)

payment_comparison["excess_captured_rs"] = (
    payment_comparison["total_captured_rs"]
    - payment_comparison["invoice_amount_rs"]
)

# Invoices where captured payments exceed the billed amount
possible_overpayments = payment_comparison[
    payment_comparison["excess_captured_rs"] > 0
].sort_values("excess_captured_rs", ascending=False)

print("Invoices with captured payments above invoice value:",
      len(possible_overpayments))

print("\nTotal possible excess captured (Rs):",
      possible_overpayments["excess_captured_rs"].sum())

print("\nLargest possible overpayments:")
print(
    possible_overpayments[
        [
            "match_key",
            "invoice_amount_rs",
            "total_captured_rs",
            "excess_captured_rs",
            "captured_payment_count",
            "payment_ids"
        ]
    ].head(30).to_string(index=False)
)


Invoices with captured payments above invoice value: 1973

Total possible excess captured (Rs): 3078429.0

Largest possible overpayments:
    match_key  invoice_amount_rs  total_captured_rs  excess_captured_rs  captured_payment_count                    payment_ids
INV2025274710             1120.0             8818.0              7698.0                     2.0 pay_0000044020, pay_000004402b
INV2026112524              330.0             7927.0              7597.0                     2.0 pay_00000680b8, pay_00000680ca
INV2026039887             4549.0            12117.0              7568.0                     2.0 pay_0000056130, pay_0000056139
INV2026138476             1670.0             9168.0              7498.0                     2.0 pay_000006e756, pay_000006e760
INV2026073945             1050.0             8497.0              7447.0                     2.0 pay_000005e7f2, pay_000005e7fb
INV2025038052              600.0             7918.0              7318.0                     2.0 pay_

In [6]:

top_invoice_ids = possible_overpayments.head(10)["match_key"]

payment_details = payments[
    (payments["match_key"].isin(top_invoice_ids)) &
    (payments["status_clean"] == "captured")
].sort_values(["match_key", "created_at_ist"])

print(payment_details[
    ["match_key", "payment_id", "amount_rs",
     "status_clean", "created_at_ist", "settlement_id"]
].to_string(index=False))


    match_key     payment_id  amount_rs status_clean            created_at_ist settlement_id
INV2025038052 pay_000000968f     7318.0     captured 2025-02-13T16:04:49+05:30 setl_67ae8780
INV2025038052 pay_00000096a8      600.0     captured 2025-02-27T20:40:50+05:30 setl_67c0fc80
INV2025129030 pay_000001fedf     2099.0     captured 2025-05-28T23:41:40+05:30 setl_6837a380
INV2025129030 pay_000001fec4     6797.0     captured 2025-05-29T16:36:44+05:30 setl_6838f500
INV2025177652 pay_000002bf9b     6997.0     captured 2025-07-25T19:09:49+05:30 setl_68841a80
INV2025177652 pay_000002bfb7     2570.0     captured 2025-07-25T20:46:34+05:30 setl_68841a80
INV2025209391 pay_0000033d98     2000.0     captured 2025-09-05T22:46:51+05:30 setl_68bb7980
INV2025209391 pay_0000033d4d     6698.0     captured 2025-09-11T22:30:42+05:30 setl_68c36280
INV2025274710 pay_0000044020     7698.0     captured 2025-11-22T18:52:28+05:30 setl_69224e80
INV2025274710 pay_000004402b     1120.0     captured 2025-11-23T01:11:

In [7]:

# Inspect the invoice and payment records for the top exceptions
check_ids = possible_overpayments.head(30)["match_key"]

# Invoice-side details
invoice_check = invoices[
    invoices["match_key"].isin(check_ids)
].copy()

print("INVOICE RECORDS")
print(invoice_check.to_string(index=False))

# Payment-side details: all statuses, not only captured
payment_check = payments[
    payments["match_key"].isin(check_ids)
].sort_values(["match_key", "created_at_ist"])

print("\nALL PAYMENT ATTEMPTS")
print(payment_check[
    ["match_key", "payment_id", "invoice_ref",
     "amount_rs", "status_clean", "created_at_ist",
     "settlement_id"]
].to_string(index=False))


INVOICE RECORDS
     invoice_id              invoice_date           booking_id      city channel  amount_rs            source_file  source_row     match_key
INV-2025-008801 2025-01-11 00:00:00+05:30 BK-HYD-202501-003814 Hyderabad  clinic     1250.0 invoices_2025Q1.csv.gz        8802 INV2025008801
INV-2025-038052 2025-02-13 00:00:00+05:30 BK-AHM-202502-007035 Ahmedabad    home      600.0 invoices_2025Q1.csv.gz       38053 INV2025038052
INV-2025-047189 2025-02-22 00:00:00+05:30 BK-PUN-202502-007603      Pune  clinic     1900.0 invoices_2025Q1.csv.gz       47190 INV2025047189
INV-2025-098009 2025-04-22 00:00:00+05:30 BK-BLR-202504-022590 Bengaluru  clinic     2949.0 invoices_2025Q2.csv.gz       18474 INV2025098009
INV-2025-129030 2025-05-28 00:00:00+05:30 BK-BLR-202505-033378 Bengaluru    home     2099.0 invoices_2025Q2.csv.gz       49495 INV2025129030
INV-2025-138738 2025-06-09 00:00:00+05:30 BK-PUN-202506-024353      Pune    home     1819.0 invoices_2025Q2.csv.gz       59203 INV20251387